# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end user authentication, live agent invocation, and observability trace extraction **without requiring any AWS access keys** from the recruiter. Credentials are obtained automatically via AWS Cognito Identity Pools.

In [ ]:
!pip install boto3 requests

import boto3
import requests
import json
import urllib.parse

access_token = None
id_token = None

### 1. Authenticate with Amazon Cognito
Recruiters authenticate as `analyst_user`. The system provides a JWT token upon success.

In [ ]:
REGION = "us-east-1"
USER_POOL_ID = "us-east-1_5pCxpIkx8"
CLIENT_ID = "2r1ik1k110jbu6nfmuoegk5lns"
IDENTITY_POOL_ID = "us-east-1:69aa806c-890a-49aa-89aa-06aaeb006691"
USERNAME = "analyst_user"
PASSWORD = "SecurePassword123!"

client = boto3.client('cognito-idp', region_name=REGION)
try:
    auth_response = client.initiate_auth(
        ClientId=CLIENT_ID,
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': USERNAME, 'PASSWORD': PASSWORD}
    )
    access_token = auth_response['AuthenticationResult']['AccessToken']
    id_token = auth_response['AuthenticationResult']['IdToken']
    print(f"✅ Cognito Authentication Successful.")
except Exception as e:
    print(f"❌ Authentication Failed: {str(e)}")

### 2. Invoke the Agentcore Streaming Endpoint
Calling the live AWS Agentcore runtime with the Bearer token.

In [ ]:
AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:162187491349:runtime/Financial_Analyst_Agent-s5aas5HZhy"
encoded_arn = urllib.parse.quote(AGENT_ARN, safe='')
AGENTCORE_URL = f"https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/{encoded_arn}/invocations"
SESSION_ID = "demo-session-recruiters-verification-proof-2026"

def query_financial_agent(prompt: str):
    if access_token is None:
        print("❌ ERROR: Access token is missing. Run Cell 1 first.")
        return

    print(f"\n--- Query: {prompt} ---")
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": SESSION_ID
    }

    response = requests.post(AGENTCORE_URL, headers=headers, json={"prompt": prompt}, stream=True)
    if response.status_code != 200:
        print(f"❌ Error {response.status_code}: {response.text}")
        return

    for line in response.iter_lines():
        if line:
            decoded = line.decode('utf-8')
            if decoded.startswith("data: "):
                data = json.loads(decoded[6:])
                print(data.get("event", ""), end="", flush=True)

### 3. Execute Required Financial Queries

In [ ]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]
for q in queries: query_financial_agent(q)

### 4. Observability Verification (Automatic AWS Identity)
This cell uses the Cognito IdToken to obtain temporary AWS credentials from the Identity Pool, allowing secure retrieval of Langfuse keys from SSM without manual credential entry.

In [ ]:
identity_client = boto3.client('cognito-identity', region_name=REGION)

try:
    # 1. Exchange IdToken for a Cognito Identity ID
    id_resp = identity_client.get_id(
        IdentityPoolId=IDENTITY_POOL_ID,
        Logins={ f'cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}': id_token }
    )
    identity_id = id_resp['IdentityId']
    
    # 2. Get Temporary AWS Credentials for the Authenticated Role
    cred_resp = identity_client.get_credentials_for_identity(
        IdentityId=identity_id,
        Logins={ f'cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}': id_token }
    )
    creds = cred_resp['Credentials']
    
    # 3. Initialize SSM client using temporary session credentials
    ssm = boto3.client(
        'ssm', 
        region_name=REGION,
        aws_access_key_id=creds['AccessKeyId'],
        aws_secret_access_key=creds['SecretKey'],
        aws_session_token=creds['SessionToken']
    )
    
    pk = ssm.get_parameter(Name='/financial-ai/langfuse/public-key', WithDecryption=True)['Parameter']['Value']
    sk = ssm.get_parameter(Name='/financial-ai/langfuse/secret-key', WithDecryption=True)['Parameter']['Value']
    
    print(f"✅ Successfully retrieved Langfuse keys using temporary AWS credentials.")
    
    trace_url = f"https://cloud.langfuse.com/api/public/traces?sessionId={SESSION_ID}"
    trace_resp = requests.get(trace_url, auth=(pk, sk))
    
    if trace_resp.status_code == 200:
        print(f"\n--- Langfuse Trace Audit ---")
        print(json.dumps(trace_resp.json(), indent=2))
    else:
        print(f"❌ Could not fetch traces: {trace_resp.status_code}")
        
except Exception as e:
    print(f"❌ Identity Exchange or Retrieval Failed: {str(e)}")